In [0]:
from datetime import datetime

def log(msg):
    """Print a timestamped log line."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

dbutils.widgets.text("landing_timestamp", "", "landing_timestamp")
dbutils.widgets.text("table_name", "", "table_name")
dbutils.widgets.text("merge_keys", "", "merge_keys")
dbutils.widgets.text("load_type", "", "load_type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
log("STEP 1: Notebook parameters received")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'" + ("  (empty -> will default to FULL)" if not load_type.strip() else ""))
print("=" * 80)

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

log("STEP 2: Reading batch from bronze layer")
print(f"  Source table : {source_table}")
print(f"  Filter       : {filter_condition}")
print("  Equivalent SQL query:")
print(f"    SELECT *\n    FROM {source_table}\n    WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"  Rows read from bronze : {bronze_count}")
print(f"  Columns ({len(bronze_df.columns)})          : {bronze_df.columns}")
if bronze_count == 0:
    log("WARNING: No rows found in bronze for this landing_timestamp - check the parameter value")
print("=" * 80)

In [0]:
from pyspark.sql import functions as F
log("STEP 3: Adding audit columns (insert_timestamp, update_timestamp)")
print("  Equivalent SQL query:")
print(f"    SELECT *,\n           current_timestamp() AS insert_timestamp,\n           current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")

updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())

print(f"  Output columns ({len(updated_df.columns)}): {updated_df.columns}")
print("=" * 80)

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"
# Empty load_type defaults to FULL to keep the original behaviour
load_type_norm = (load_type or "FULL").strip().upper()

cols_list = ", ".join([f"`{c}`" for c in updated_df.columns])
table_exists = spark.catalog.tableExists(target_table)

log("STEP 4: Writing to silver layer")
print(f"  Target table      : {target_table}")
print(f"  Load type (input) : '{load_type}'  ->  resolved to: {load_type_norm}")
print(f"  Target exists     : {table_exists}")
if table_exists:
    rows_before = spark.read.table(target_table).count()
    print(f"  Rows in target BEFORE write : {rows_before}")

if load_type_norm == "FULL":
    # Full load: overwrite the target table
    log("  FULL load -> overwriting target table with the current batch")
    print("  Equivalent SQL query:")
    print(f"    CREATE OR REPLACE TABLE {target_table} AS\n    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")
    updated_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

elif load_type_norm == "APPEND":
    # Append: add the new batch to the target table (creates it if missing)
    log("  APPEND load -> adding batch to target table" + ("" if table_exists else " (table will be created)"))
    print("  Equivalent SQL query:")
    print(f"    INSERT INTO {target_table} ({cols_list})\n    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type_norm == "MERGE":
    # Merge (upsert): update existing rows and insert new ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    log(f"  MERGE load -> merge keys parsed: {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")

    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")

    if not spark.catalog.tableExists(target_table):
        # First run: nothing to merge into, so create the table
        log("  Target table does not exist -> first run, creating it from the batch instead of merging")
        print("  Equivalent SQL query:")
        print(f"    CREATE TABLE {target_table} AS\n    SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n    FROM {source_table}\n    WHERE {filter_condition}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # Keep the original insert_timestamp on updates; refresh everything else
        update_set = {f"`{c}`": f"s.`{c}`" for c in updated_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in updated_df.columns}

        update_clause = ",\n      ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols = ", ".join(insert_set.keys())
        insert_vals = ", ".join(insert_set.values())
        print(f"  Merge condition   : {merge_condition}")
        print(f"  Columns updated   : {len(update_set)} (insert_timestamp preserved on update)")
        print(f"  Columns inserted  : {len(insert_set)}")
        print("  Equivalent SQL query:")
        print(f"    MERGE INTO {target_table} AS t\n"
              f"    USING (\n"
              f"      SELECT *, current_timestamp() AS insert_timestamp, current_timestamp() AS update_timestamp\n"
              f"      FROM {source_table}\n"
              f"      WHERE {filter_condition}\n"
              f"    ) AS s\n"
              f"    ON {merge_condition}\n"
              f"    WHEN MATCHED THEN UPDATE SET\n      {update_clause}\n"
              f"    WHEN NOT MATCHED THEN INSERT ({insert_cols})\n"
              f"      VALUES ({insert_vals})")
        log("  Executing MERGE...")

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(updated_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute())

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

# Post-write summary from the Delta transaction log
rows_after = spark.read.table(target_table).count()
print(f"  Rows in target AFTER write  : {rows_after}")
last_op = DeltaTable.forName(spark, target_table).history(1).select("version", "operation", "operationMetrics").collect()[0]
print(f"  Delta version written       : {last_op['version']}")
print(f"  Delta operation             : {last_op['operation']}")
print("  Operation metrics:")
for k, v in sorted((last_op["operationMetrics"] or {}).items()):
    print(f"    {k:<35}: {v}")

log(f"{load_type_norm} load completed for {target_table}")
print("=" * 80)

In [0]:
record_count = updated_df.count()
log("STEP 5: Returning result to caller (ADF)")
print(f"  Records processed in this batch : {record_count}")
print(f"  Exiting notebook with value     : '{record_count}'")
print("=" * 80)
dbutils.notebook.exit(str(record_count))